# Lesson 15 — 给提醒器做一个 HTTP 控制接口

这节课仍然延续桌面状态提醒器，但数据首次穿过**真实本机网络**。不需要 ESP32，也不需要 Python 基础：下面代码格是完整 C++ 程序。

## 我们要解决什么问题？

电脑怎样用浏览器友好的方式读取并修改提醒器？

## 最小知识

HTTP 把 method、path、headers 和 body 组织成请求；REST 只是围绕资源设计接口的约定。JSON 是数据格式，不等于网络协议。这个微型 server 只实现本课所需的最小子集。

运行初始化格后，所有网络实验只绑定 `127.0.0.1`。Notebook 面板是真实客户端；业务逻辑、协议响应和状态都来自当前 C++ 代码格。

In [ ]:
# 运行支架：只执行一次。你写的网络逻辑仍是 C++，不需要学习 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))
from network_lab import register
register()


## 1. 第一个可运行实验

先 GET 状态，再 POST 开启/关闭，然后再 GET。面板显示真实 HTTP status、content type 和 JSON body；C++ 决定响应内容与状态变化。

运行下一格。面板的端口由 C++ 程序通过 `ACADEMY_READY` 报告；不要手填固定端口。

In [ ]:
%%network_lab l15-guided http guided pass
#include "academy/net/http.hpp"

int main() {
    bool reminder_on = false;
    return academy::net::run_http_server([&](const academy::net::HttpRequest& request) {
        using academy::net::HttpResponse;
        if (request.method == "GET" && request.path == "/api/status") {
            return HttpResponse{200, "application/json; charset=utf-8",
                                std::string("{\"reminder\":") + (reminder_on ? "true}" : "false}")};
        }
        if (request.method == "POST" && request.path == "/api/reminder") {
            if (request.body == "{\"enabled\":true}") reminder_on = true;
            else if (request.body == "{\"enabled\":false}") reminder_on = false;
            else return HttpResponse{400, "application/json; charset=utf-8", "{\"error\":\"enabled expected\"}"};
            return HttpResponse{200, "application/json; charset=utf-8",
                                std::string("{\"reminder\":") + (reminder_on ? "true}" : "false}")};
        }
        return HttpResponse{404, "application/json; charset=utf-8", "{\"error\":\"not found\"}"};
    });
}


## 2. 修改一次

把下一格中一个响应文本或状态初值改掉，再点击面板操作，观察改变从 C++ 穿过真实 localhost 连接回到输出。重新运行本格会终止旧进程并启动新进程。

In [ ]:
%%network_lab l15-modify http
#include "academy/net/http.hpp"

int main() {
    bool reminder_on = false;
    return academy::net::run_http_server([&](const academy::net::HttpRequest& request) {
        using academy::net::HttpResponse;
        if (request.method == "GET" && request.path == "/api/status") {
            return HttpResponse{200, "application/json; charset=utf-8",
                                std::string("{\"reminder\":") + (reminder_on ? "true}" : "false}")};
        }
        if (request.method == "POST" && request.path == "/api/reminder") {
            if (request.body == "{\"enabled\":true}") reminder_on = true;
            else if (request.body == "{\"enabled\":false}") reminder_on = false;
            else return HttpResponse{400, "application/json; charset=utf-8", "{\"error\":\"enabled expected\"}"};
            return HttpResponse{200, "application/json; charset=utf-8",
                                std::string("{\"reminder\":") + (reminder_on ? "true}" : "false}")};
        }
        return HttpResponse{404, "application/json; charset=utf-8", "{\"error\":\"not found\"}"};
    });
}


## 3. 故意弄坏

下面代码可编译，但**自动检查应失败**。先运行，点击“播放自动检查”，再根据红色结果解释问题。失败来自真实网络往返后的行为差异，不是 Notebook 预设动画。

In [ ]:
%%network_lab l15-broken http exercise fail
#include "academy/net/http.hpp"

int main() {
    bool reminder_on = false;
    return academy::net::run_http_server([&](const academy::net::HttpRequest& request) {
        using academy::net::HttpResponse;
        if (request.method == "GET" && request.path == "/api/status") {
            return HttpResponse{200, "application/json", std::string("{\"reminder\":") + (reminder_on ? "true}" : "false}")};
        }
        if (request.method == "POST" && request.path == "/api/reminder") {
            bool reminder_on = request.body == "{\"enabled\":true}"; // 遮蔽了设备状态。
            return HttpResponse{200, "application/json", reminder_on ? "{\"reminder\":true}" : "{\"reminder\":false}"};
        }
        return HttpResponse{404, "application/json", "{}"};
    });
}


## 4. 自己完成

在下方 C++ 中完成 TODO。先手动操作，再点“播放自动检查”。测试只编译这一格，不会读取 `solution/`。需要时再看 [分层 Hint](../lessons/15-http-rest-json/hints.md)。

In [ ]:
%%network_lab l15-exercise http exercise
#include "academy/net/http.hpp"

int main() {
    bool reminder_on = false;
    return academy::net::run_http_server([&](const academy::net::HttpRequest& request) {
        using academy::net::HttpResponse;
        // TODO: GET /api/status 返回 {"reminder":false/true}。
        // TODO: POST /api/reminder 接受 {"enabled":true/false} 并保存状态。
        (void)request;
        (void)reminder_on;
        return HttpResponse{404, "application/json; charset=utf-8", "{\"error\":\"TODO\"}"};
    });
}


## 5. 你刚刚真正学到了什么？

你已经让提醒器通过真实 localhost 网络与另一个进程通信，并使用协议契约验证状态。上层的 URL、JSON 与状态机可迁移到 ESP32 的 HTTP server adapter。真实设备要限制请求体、避免阻塞传感器任务，并添加认证或至少局域网边界说明。

完成尝试后才查看 [Solution Notebook](solutions/15-http-rest-json-solution.ipynb)。